# 01 — Implementing U-Net from Scratch

**Goal:** understand every stage of the encoder-decoder and verify tensor dimensions.

This implementation uses basic PyTorch layers: convolutions, batch normalization, ReLU, max pooling, transposed convolutions and concatenation. It does not load a pretrained segmentation network.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
import torch
from src.model import UNet, count_parameters

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels=3, num_classes=3, base_channels=16).to(device)
print("Device:", device)
print("Trainable parameters:", f"{count_parameters(model):,}")

## 1. Encoder

Each `DoubleConv` block applies two 3×3 convolutions. Batch normalization stabilizes feature distributions and ReLU introduces non-linearity. Max pooling halves height and width.

For a 128×128 input, the encoder resolutions are 128, 64, 32 and 16. The bottleneck operates at 8×8.

In [ ]:
x = torch.randn(2, 3, 128, 128).to(device)
with torch.no_grad():
    e1 = model.enc1(x)
    e2 = model.enc2(model.pool(e1))
    e3 = model.enc3(model.pool(e2))
    e4 = model.enc4(model.pool(e3))
    b = model.bottleneck(model.pool(e4))
print("Input:", tuple(x.shape))
for name, tensor in [("encoder 1", e1), ("encoder 2", e2),
                     ("encoder 3", e3), ("encoder 4", e4),
                     ("bottleneck", b)]:
    print(name, tuple(tensor.shape))

## 2. Decoder and skip connections

The decoder upsamples a coarse feature map. It concatenates the result with the encoder feature map at the same spatial resolution.

The encoder path preserves local information that may be weakened by repeated downsampling. The decoder path carries deeper context. Concatenation gives the decoder access to both.

In [ ]:
with torch.no_grad():
    d4 = model.up4(b)
    cat4 = torch.cat([d4, e4], dim=1)
    d4 = model.dec4(cat4)
    d3 = model.up3(d4)
    cat3 = torch.cat([d3, e3], dim=1)
    d3 = model.dec3(cat3)
    d2 = model.up2(d3)
    cat2 = torch.cat([d2, e2], dim=1)
    d2 = model.dec2(cat2)
    d1 = model.up1(d2)
    cat1 = torch.cat([d1, e1], dim=1)
    d1 = model.dec1(cat1)
    logits = model.head(d1)
for name, tensor in [("upsampled 4", cat4), ("decoder 4", d4),
                     ("upsampled 3", cat3), ("decoder 3", d3),
                     ("upsampled 2", cat2), ("decoder 2", d2),
                     ("upsampled 1", cat1), ("decoder 1", d1),
                     ("output logits", logits)]:
    print(name, tuple(tensor.shape))

## 3. Why logits?

The output has three channels, one for each class. Each pixel has three raw scores. The training loss applies the appropriate normalization internally. At inference, the class with the largest logit is selected.

The model returns `[batch, classes, height, width]`.

In [ ]:
with torch.no_grad():
    output = model(torch.randn(1, 3, 128, 128).to(device))
    predicted_mask = output.argmax(dim=1)
print("Logits:", tuple(output.shape))
print("Predicted mask:", tuple(predicted_mask.shape))
print("Predicted class IDs:", torch.unique(predicted_mask).cpu().tolist())

## 4. Your architecture explanation

In your own words, explain:
1. What a feature map represents.
2. Why spatial resolution decreases in the encoder.
3. What the bottleneck contributes.
4. How transposed convolution upsamples.
5. Why encoder and decoder feature maps are concatenated.
6. Why the final layer uses a 1×1 convolution.

**Deliverable:** export or screenshot the tensor-shape output and include it in your README.